### Load survey

In [ ]:
import pandas as pd
df = pd.read_csv('D2.csv')
print(df.info())

### Categorical distributions

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

categorical_data = ['gender', 'age', 'blood_type', 'insurance', 'income', 'smoking', 'working']

for i in categorical_data:
    fig, ax = plt.subplots(figsize=(3, 2))
    sns.countplot(data=df, x=i)  
    ax.set_xlabel(i) 
    plt.xticks(rotation=45)
    plt.show()

### Numerical distributions

In [ ]:
numerical_variables = ['height', 'weight', 'contacts_count', 'worried', 'alcohol', 'covid19_positive']

plt.figure(figsize=(15, 10))

for i, variable in enumerate(numerical_variables):
    plt.subplot(3, 2, i + 1)
    sns.histplot(df[variable], kde=True,)
        
plt.tight_layout()
plt.show()

### Correlation matrix

In [ ]:
numerical_variables = ['height', 'weight', 'contacts_count', 'worried', 'alcohol', 'covid19_positive']

df_numeric = df[numerical_variables]
corr_matrix = df_numeric.corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', annot_kws={"size": 10})
plt.show()

### Select features

In [ ]:
df2 = df.copy()
df2 = df2[["gender", "weight","worried", "income", "smoking", "working", "insurance", "covid19_positive"]]

### Inspect types

In [ ]:
print(df2.info())

### Encode categories

In [ ]:
gender_map = {'male': 1, 'female': 2, 'other': 3}
income_map = {'low': 1, 'med': 2, 'high': 3, 'gov': 4}
insurance_map = {'no': 0, 'yes': 1}
smoking_map = {'never': 0, 'yesmedium': 1, 'quit0': 2, 'quit10': 3, 'quit5': 4, 'yesheavy': 5, 'yeslight': 6, 'vape': 7}
working_map = {'never': 1, 'stopped': 2, 'travel non critical': 3, 'travel critical': 4, 'home': 5}

df2['gender'] = df2['gender'].map(gender_map)
df2['income'] = df2['income'].map(income_map)
df2['insurance'] = df2['insurance'].map(insurance_map)
df2['smoking'] = df2['smoking'].map(smoking_map)
df2['working'] = df2['working'].map(working_map)
print(df2.info())

### Scale numerical features

In [ ]:
from sklearn.preprocessing import StandardScaler
X = df2.drop(columns=['covid19_positive']).to_numpy(dtype=float)
scaler = StandardScaler()
X[:, [1, 2]] = scaler.fit_transform(X[:, [1, 2]])

### Inspect data

In [ ]:
print(df2.head())

### Elbow method

In [ ]:
from kmodes.kprototypes import KPrototypes
import matplotlib.pyplot as plt

rs = 42
# list to save the clusters and cost 
clusters = []
cost_vals = []

for k in range(2, 10, 2):
    model = KPrototypes(n_clusters=k, random_state=rs)
    model.fit_predict(X, categorical=[0, 3, 4, 5, 6])
    clusters.append(model)
    cost_vals.append(model.cost_)

plt.plot(range(2, 10, 2), cost_vals, marker='*')
plt.show()

### Mixed silhouette

In [ ]:
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import euclidean_distances
from scipy.spatial.distance import cdist
import numpy as np
sil_scores = []
# Use the same mixed dissimilarity as the fitted K-Prototypes objective.
idx = np.random.default_rng(rs).choice(len(X), min(1500, len(X)), replace=False)
for k, model in zip(range(2, 10, 2), clusters):
    X_num = X[idx][:, [1, 2]]
    X_cat = X[idx][:, [0, 3, 4, 5, 6]]
    distance = euclidean_distances(X_num, squared=True) + model.gamma * cdist(X_cat, X_cat, metric='hamming') * X_cat.shape[1]
    np.fill_diagonal(distance, 0)
    silScore = silhouette_score(distance, model.labels_[idx], metric='precomputed') if hasattr(model, 'labels_') else silhouette_score(distance, model.predict(X, categorical=[0, 3, 4, 5, 6])[idx], metric='precomputed')
    sil_scores.append(silScore)
    print(f"Mixed silhouette for k={k}: {silScore:.4f}")
plt.plot(range(2, 10, 2), sil_scores, marker='*')
plt.show()

### Fit and profile clusters

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from kmodes.kprototypes import KPrototypes

optimal_k = list(range(2, 10, 2))[int(np.argmax(sil_scores))]
model = KPrototypes(n_clusters=optimal_k, random_state=rs)
y = model.fit_predict(X, categorical=[0, 3, 4, 5, 6])
df2['Cluster_ID'] = y

print("Cluster membership")
print(df2['Cluster_ID'].value_counts())

custom_palette = dict(zip(range(optimal_k), sns.color_palette('tab10', optimal_k)))

print(df2.groupby('Cluster_ID')['covid19_positive'].agg(['size', 'mean']))
sns.pairplot(df2, vars=['weight', 'worried'], hue='Cluster_ID', diag_kind='hist', palette=custom_palette)
plt.show()

### Cluster distributions

In [ ]:
cols = ['gender', 'weight', 'worried', 'income', 'smoking', 'working', 'insurance', 'covid19_positive']
n_bins = 20
clusters_to_inspect = sorted(df2.Cluster_ID.unique())

for cluster in clusters_to_inspect:
    print("Distribution for Cluster {}".format(cluster))
    fig, axs = plt.subplots(nrows=len(cols), ncols=1, figsize=(10, 2 * len(cols))) 
    axs = axs.flatten()
    fig.suptitle("Cluster {}".format(cluster))
    
    for i, col in enumerate(cols):
        bin_range = (min(df2[col]), max(df2[col]))
        sns.histplot(data=df2[df2['Cluster_ID'] == cluster], x=col, bins=n_bins, binrange=bin_range, kde=col in ['weight', 'worried'], ax=axs[i])
        axs[i].set_title(col)

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

### Add age

In [ ]:
df2['age'] = df['age']

age_map = {
    '0_10': 0, '10_20': 1, '20_30': 2, '30_40': 3, '40_50': 4,
    '50_60': 5, '60_70': 6, '70_80': 7, '80_90': 8, '90_100': 9, '100_110': 10
}
df2['age'] = df2['age'].map(age_map)
df3 = df2.drop(columns=['Cluster_ID']).copy()
print(df2.info())

### Age sensitivity

In [ ]:
from sklearn.metrics import adjusted_rand_score
X_age = df3.drop(columns=['covid19_positive']).to_numpy(dtype=float)
X_age[:, [1, 2]] = scaler.transform(X_age[:, [1, 2]])
categorical_features = [0, 3, 4, 5, 6, 7]
model = KPrototypes(n_clusters=optimal_k, random_state=42)
y = model.fit_predict(X_age, categorical=categorical_features)
df3['Cluster_ID'] = y
print(df3['Cluster_ID'].value_counts())
print('Adjusted Rand Index:', adjusted_rand_score(df2.Cluster_ID, y))
print(df3.groupby('Cluster_ID')['covid19_positive'].agg(['size', 'mean']))
sns.pairplot(df3, vars=['weight', 'worried'], hue='Cluster_ID', diag_kind='hist', palette=custom_palette)
plt.show()

### Age profiles

In [ ]:
clusters_to_inspect = df3['Cluster_ID'].unique()
cols = ['gender', 'weight', 'worried', 'income', 'smoking', 'working', 'insurance', 'covid19_positive','age']
n_bins = 20

for cluster in clusters_to_inspect:
    print(f"Distribution for Cluster {cluster}")
    fig, axs = plt.subplots(nrows=len(cols), ncols=1, figsize=(10, 2 * len(cols))) 
    fig.suptitle(f'Cluster {cluster}')
    for i, col in enumerate(cols):
        bin_range = (min(df3[col]), max(df3[col]))
        sns.histplot(data=df3[df3['Cluster_ID'] == cluster], x=col, bins=n_bins, binrange=bin_range, kde=col in ['weight', 'worried'], ax=axs[i], color='orange')
        axs[i].set_title(col)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()